# Deep Agents Commercial Assistant — Demo

Demonstrates the **Deep Agents** implementation: a high-level declarative agent built
with `create_deep_agent()`. No manual `StateGraph`, no explicit nodes or edges —
just tools, a system prompt, and an `interrupt_on` declaration.

## Scenarios
1. List leads
2. Add a new lead
3. Guardrail — out-of-scope question
4. HITL — status update → `qualified` (approve flow)
5. HITL — email draft (cancel flow)
6. Conversation memory — same thread across multiple turns

Compare with the LangGraph equivalent: `langgraph/demo.ipynb`.

> **Key difference in scenario 4:** Deep Agents interrupts on *all* calls to
> `update_lead_status_tool`, not only when `new_status='lost'`. This is the
> trade-off of the declarative `interrupt_on` approach: less granular, but
> zero boilerplate.

## Setup

In [ ]:
import os

from dotenv import find_dotenv, load_dotenv

load_dotenv(find_dotenv())  # searches parent directories for .env

# ── LangSmith tracing ─────────────────────────────────────────────────────────
os.environ["LANGCHAIN_TRACING_V2"] = "true"
os.environ["LANGCHAIN_PROJECT"] = "lab-06-deep-agents"

# ── Provider / model ──────────────────────────────────────────────────────────
PROVIDER = "anthropic"        # "anthropic" | "openai" | "google"
MODEL    = "claude-sonnet-4-6"  # None → uses provider default

print(f"Provider : {PROVIDER}")
print(f"Model    : {MODEL}")
print(f"Project  : {os.environ['LANGCHAIN_PROJECT']}")

In [ ]:
from langchain_core.messages import HumanMessage
from langgraph.types import Command


def show(messages):
    """Print conversation messages in a readable format."""
    for msg in messages:
        role = type(msg).__name__.replace("Message", "")
        content = msg.content
        # Normalize Google Gemini's structured content blocks
        if isinstance(content, list):
            content = " ".join(
                b.get("text", "") for b in content
                if isinstance(b, dict) and b.get("type") == "text"
            )
        if hasattr(msg, "tool_calls") and msg.tool_calls:
            for tc in msg.tool_calls:
                print(f"[{role:8}] → {tc['name']}({tc['args']})")
        if content:
            preview = content[:500] + ("…" if len(content) > 500 else "")
            print(f"[{role:8}] {preview}")

## Agent Architecture

The Deep Agents agent is configured declaratively in ~15 lines:

```python
HITL_TOOLS = {
    "generate_email_draft_tool": True,
    "update_lead_status_tool": True,   # ALL status changes — no conditional logic
}

agent = create_deep_agent(
    model="anthropic:claude-sonnet-4-6",
    tools=TOOLS,
    system_prompt=SYSTEM_PROMPT,
    interrupt_on=HITL_TOOLS,
    checkpointer=MemorySaver(),
)
```

Compare with the explicit `StateGraph`, `human_review_node`, and `_find_hitl_call()` function
needed in the LangGraph version (~150 lines vs ~70 here).

In [ ]:
from deep_agents.agent.agent import HITL_TOOLS, create_agent

print("HITL_TOOLS (declarative interrupt_on config):")
for tool_name, enabled in HITL_TOOLS.items():
    print(f"  {tool_name}: {enabled}")

print()
agent = create_agent(provider=PROVIDER, model=MODEL)
print("Agent compiled ✓")

## Scenario 1 — List leads

In [ ]:
config_1 = {"configurable": {"thread_id": "demo-da-list"}}

result = agent.invoke(
    {"messages": [HumanMessage(content="Show me all my leads.")]},
    config_1,
)
show(result["messages"])

## Scenario 2 — Add a lead

In [ ]:
config_2 = {"configurable": {"thread_id": "demo-da-add"}}

result = agent.invoke(
    {
        "messages": [
            HumanMessage(
                content=(
                    "Add a new lead: Jean Dupont from StartupX, "
                    "email j.dupont@startupx.io"
                )
            )
        ]
    },
    config_2,
)
show(result["messages"])

## Scenario 3 — Guardrail

The system prompt restricts the agent to sales pipeline topics.
Out-of-scope questions are refused without calling any tool.

In [ ]:
config_3 = {"configurable": {"thread_id": "demo-da-guardrail"}}

result = agent.invoke(
    {"messages": [HumanMessage(content="What is the capital of France?")]},
    config_3,
)
show(result["messages"])

## Scenario 4 — HITL: Status update → `qualified` (approve)

Unlike the LangGraph agent, Deep Agents interrupts on **all** calls to
`update_lead_status_tool` — including harmless transitions like `prospect → qualified`.
This is the trade-off of the declarative `interrupt_on` approach.

The interrupt payload format is also different:
```json
{"action_requests": [{"name": "update_lead_status_tool", "args": {...}}]}
```
vs the LangGraph format (`{"tool_name": ..., "tool_args": ..., "message": ...}`).

In [ ]:
config_4 = {"configurable": {"thread_id": "demo-da-hitl-status"}}

# Stream until the agent pauses
list(
    agent.stream(
        {"messages": [HumanMessage(content="Mark lead_001 as qualified.")]},
        config_4,
        stream_mode="values",
    )
)

state = agent.get_state(config_4)
print("Paused at node:", state.next)

# Deep Agents interrupt format
interrupt_val = state.tasks[0].interrupts[0].value
print("\nInterrupt payload (Deep Agents format):")
if "action_requests" in interrupt_val:
    req = interrupt_val["action_requests"][0]
    print(f"  tool : {req['name']}")
    print(f"  args : {req['args']}")

In [ ]:
# Approve — Deep Agents v2 resume format
result = agent.invoke(
    Command(resume={"decisions": [{"type": "approve"}]}),
    config_4,
)
show(result["messages"][-3:])

## Scenario 5 — HITL: Email draft (cancel)

`generate_email_draft_tool` always triggers HITL — same behaviour as LangGraph.
Here we reject the action.

In [ ]:
config_5 = {"configurable": {"thread_id": "demo-da-hitl-email"}}

list(
    agent.stream(
        {
            "messages": [
                HumanMessage(content="Generate a follow-up email for lead_003.")
            ]
        },
        config_5,
        stream_mode="values",
    )
)

state = agent.get_state(config_5)
interrupt_val = state.tasks[0].interrupts[0].value
print("Pending action (Deep Agents format):")
if "action_requests" in interrupt_val:
    req = interrupt_val["action_requests"][0]
    print(f"  tool : {req['name']}")
    print(f"  args : {req['args']}")

In [ ]:
# Reject — equivalent to LangGraph's "cancel"
result = agent.invoke(
    Command(resume={"decisions": [{"type": "reject"}]}),
    config_5,
)
show(result["messages"][-3:])

## Scenario 6 — Conversation memory

Same `MemorySaver` checkpointing as LangGraph — state persists across invocations
on the same `thread_id`.

> State is in-memory only — it resets on kernel restart.

In [ ]:
config_6 = {"configurable": {"thread_id": "demo-da-memory"}}

# Turn 1
result = agent.invoke(
    {"messages": [HumanMessage(content="List all prospect leads.")]},
    config_6,
)
print("=== Turn 1 ===")
show(result["messages"][-2:])

# Turn 2 — agent references the previous response
result = agent.invoke(
    {"messages": [HumanMessage(content="How many leads did you just show me?")]},
    config_6,
)
print("\n=== Turn 2 ===")
show(result["messages"][-2:])

## LangSmith Traces

All runs above were sent to the **`lab-06-deep-agents`** project.

Open [LangSmith](https://smith.langchain.com) → select the project to inspect:
- Token usage and cost per run
- Latency breakdown
- Tool call sequence
- Full message history

These traces will be compared against the `lab-06-langgraph` traces in `comparison.ipynb`.

In [ ]:
print(f"LangSmith project : {os.environ['LANGCHAIN_PROJECT']}")
print("URL               : https://smith.langchain.com")